# 美团歪马送酒 · 采销中台数据分析与决策算力 Pipeline

**说明**：本 Notebook 专为美团歪马送酒采销中台业务分析师（BA）设计，包含 DWD/DWS 数仓取数 Hive SQL 模板、DuPont 结构归因、动态季节窗四象限、价格弹性 $Ed$ 回归算法、PSM 补贴误伤剥离以及供应商 100 分制评分卡全套 Python 生产级代码。

## 1. 基础环境与数据加载

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

# 模拟商品数据加载 (已过滤单笔订单 > 50箱 或 >10000元 的团购大单噪音)
data = {
    'product_id': ['SKU1001', 'SKU1002', 'SKU1003', 'SKU1004', 'SKU1005', 'SKU1006'],
    'product_name': ['青岛纯生500ml*12', '经典雪花500ml*12', '普五52度500ml', '水晶剑52度500ml', '哈尔滨特醇330ml', '飞天茅台53度'],
    'sales_30d': [48500, 42000, 68000, 38000, 32000, 14500],
    'gross_margin': [0.345, 0.333, 0.190, 0.264, 0.100, 0.276],
    'days_on_shelf': [240, 310, 400, 180, 120, 600],
    'sell_through': [0.85, 0.92, 0.65, 0.78, 0.88, 0.40]
}
df_products = pd.DataFrame(data)
print('--> 商品基础明细表加载成功:')
print(df_products)

## 2. DuPont 因素贡献度 + 品类结构归因算法

In [ ]:
def dupont_structure_attribution(traffic_0, traffic_1, cvr_0, cvr_1, aov_0, aov_1, mix_shift_impact=15000):
    contrib_traffic = (traffic_1 - traffic_0) * cvr_0 * aov_0
    contrib_cvr = traffic_0 * (cvr_1 - cvr_0) * aov_0
    contrib_aov = traffic_0 * cvr_0 * (aov_1 - aov_0)
    total_delta = contrib_traffic + contrib_cvr + contrib_aov + mix_shift_impact
    
    return {
        '流量变动贡献(元)': round(contrib_traffic, 2),
        '转化率变动贡献(元)': round(contrib_cvr, 2),
        '客单价变动贡献(元)': round(contrib_aov, 2),
        '品类结构偏移贡献(元)': round(mix_shift_impact, 2),
        'GMV总变动量(元)': round(total_delta, 2)
    }

# 示例运行: 流量减少2500, CVR下滑0.3%, 客单价下滑1.5元
res_attribution = dupont_structure_attribution(50000, 47500, 0.0512, 0.0482, 118.5, 117.0)
for k, v in res_attribution.items():
    print(f"{k}: {v:,.2f}")

## 3. 动态季节时间窗 (Season Flag) 四象限切分算法

In [ ]:
def calc_quadrant_seasonal(df, season_flag='SUMMER'):
    # 夏季模式降低销量中位数门槛看周转，冬季重视毛利
    q_sales = 0.45 if season_flag == 'SUMMER' else 0.50
    q_margin = 0.50 if season_flag == 'SUMMER' else 0.55
    
    sales_p50 = df['sales_30d'].quantile(q_sales)
    margin_p50 = df['gross_margin'].quantile(q_margin)
    
    conditions = [
        (df['sales_30d'] >= sales_p50) & (df['gross_margin'] >= margin_p50),
        (df['sales_30d'] >= sales_p50) & (df['gross_margin'] < margin_p50),
        (df['sales_30d'] < sales_p50) & (df['gross_margin'] >= margin_p50),
        (df['sales_30d'] < sales_p50) & (df['gross_margin'] < margin_p50)
    ]
    choices = ['规模品', '流量品', '利润品', '培育品']
    df['product_role'] = np.select(conditions, choices, default='培育品')
    return df, sales_p50, margin_p50

df_tagged, s_p50, m_p50 = calc_quadrant_seasonal(df_products, season_flag='SUMMER')
print(f"--> 夏季模式切分门槛: 销售额P50=¥{s_p50:,.2f}, 毛利率P50={m_p50*100:.1f}%")
print(df_tagged[['product_id', 'product_name', 'product_role']])

## 4. 价格弹性 $Ed$ 与竞对锚点约束售价 $P^*$

In [ ]:
def optimal_price_with_competitor(mc, elasticity, competitor_price=None):
    abs_ed = abs(elasticity)
    if abs_ed <= 1.0:
        p_star = mc * 1.45
    else:
        p_star = mc * (abs_ed / (abs_ed - 1.0))
    
    final_price = p_star
    anchor_triggered = False
    if competitor_price and competitor_price > 0:
        max_cap = competitor_price * 1.05
        if p_star > max_cap:
            final_price = max_cap
            anchor_triggered = True
            
    return round(p_star, 2), round(final_price, 2), anchor_triggered

# 示例: 边际成本 35元, 弹性 -2.3, 竞对售价 58元
p_star, final_price, triggered = optimal_price_with_competitor(35.0, -2.3, competitor_price=58.0)
print(f"理论售价 P*: ¥{p_star}, 竞对约束后售价: ¥{final_price}, 触发溢价封顶: {triggered}")